**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 12c: Unsloth 기반 파인튜닝 (LoRA 최적화 비교)

---

### 📋 학습 목표

**Unsloth**는 LLM 파인튜닝을 **2배 빠르게**, **60% 적은 메모리**로 수행할 수 있게 해주는 최적화 라이브러리입니다.

본 세션은 Session 12a(이론)·12b(LoRA vs FFT 비교)에서 다룬 LoRA를 **Unsloth로 가속**할 때 얼마나 빨라지는지를 **동일 모델·동일 데이터·동일 LoRA 설정**으로 실측 비교합니다.

- 🔹 동일 조건에서 Unsloth vs 일반 Transformers 학습 시간·VRAM 실측 비교
- 🔹 `FastLanguageModel`로 모델 로드 + 양자화 + LoRA를 한 번에 적용
- 🔹 `save_pretrained_gguf`로 GGUF 변환 후 Ollama 연동
- 🔹 Unsloth 최적화 원리 이해 (커스텀 CUDA 커널, RoPE 캐싱, in-place CE Loss)

### ⚠️ 하드웨어 요구사항 — 반드시 확인하세요

Unsloth는 **Ampere(SM 8.0)+ GPU에서만 실행 가능**합니다.

| GPU | 아키텍처 | SM | Unsloth 실행 |
|-----|----------|-----|--------------|
| RTX 4090 / 4080 / 4070 / 4060 | Ada Lovelace | 8.9 | ✅ |
| RTX 3090 / 3080 / 3070 / 3060 | Ampere | 8.6 | ✅ |
| A100 / A6000 / A40 | Ampere | 8.0 | ✅ |
| **RTX 2060 / 2070 / 2080 / TITAN RTX** | **Turing** | **7.5** | **❌** |
| GTX 1060 / 1070 / 1080 (Pascal) | Pascal | 6.1 | ❌ |

**Turing(SM 7.5) 이하 GPU에서는 방법 B(Unsloth) 비교 실험이 자동으로 스킵되고, 방법 A(일반 Transformers QLoRA)만 실행됩니다.** 본 노트북의 개념과 코드는 그대로 학습 가치가 있으며, Ampere+ 환경(클라우드 GPU 등)에서 재실행하시면 비교까지 가능합니다.

### 📦 필요 라이브러리

```
unsloth, transformers, trl, peft, bitsandbytes, datasets, torch
```

### ⏱️ 예상 소요 시간: 약 90분 (비교 실험 포함 시)

### 🔗 선행 학습
- **Session 12a**: LoRA 이론
- **Session 12b**: LoRA vs FFT 실전 비교

---

## 1️⃣ Unsloth 설치 및 설정

⚠️ Unsloth는 별도 설치가 필요합니다. 이미 설치되어 있지 않다면 아래 셀을 실행하세요.

In [ ]:
# Unsloth 설치 (필요시 주석 해제)
# !pip install unsloth
# 또는 특정 버전
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("⚠️ Unsloth가 설치되어 있지 않다면 위의 주석을 해제하고 실행하세요.")
print("📌 설치 후 런타임을 재시작해야 할 수 있습니다.")

In [ ]:
import torch
import gc
import os
import json
import time
import warnings
import importlib
warnings.filterwarnings('ignore')

# GPU 메모리 모니터링
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

# 환경 확인
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

gpu_name = torch.cuda.get_device_name(0)
gpu_cap = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} (compute {gpu_cap[0]}.{gpu_cap[1]})")
print_gpu_memory("시작")

# Unsloth 비교 가능 여부 판단
# ⚠️ Unsloth는 import 시 SFTTrainer를 패치하므로 여기서는 import하지 않음
UNSLOTH_INSTALLED = importlib.util.find_spec('unsloth') is not None
BF16_SUPPORTED = gpu_cap[0] >= 8  # Ampere 이상

if UNSLOTH_INSTALLED and BF16_SUPPORTED:
    print("\n✅ Unsloth 설치 확인 + Ampere+ GPU")
    print("   → 일반 Transformers vs Unsloth 비교 실험 진행!")
    CAN_COMPARE = True
elif not BF16_SUPPORTED:
    print(f"\n⚠️ GPU compute {gpu_cap[0]}.{gpu_cap[1]} → Unsloth 사용 불가 (Ampere 8.0+ 필요)")
    print("   → 일반 Transformers로만 진행합니다.")
    CAN_COMPARE = False
else:
    print("\n⚠️ Unsloth가 설치되어 있지 않습니다.")
    print("   📌 pip install unsloth 후 커널 재시작 필요")
    print("   → 일반 Transformers로만 진행합니다.")
    CAN_COMPARE = False

## 2️⃣ 공통 설정 (모델·데이터·평가 프롬프트)

두 방법 모두 동일한 조건으로 실험하기 위해 공통 설정을 먼저 준비합니다.

In [ ]:
# 공통 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # 비교 실험용 통일 모델
MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

# 토크나이저 로드
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 데이터 로드 + 포맷팅
from datasets import Dataset

data_path = "../data/samples/alpaca_ko_sample.json"
with open(data_path, "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

def format_to_chat(sample):
    instruction = sample["instruction"]
    input_text = sample.get("input", "")
    output_text = sample["output"]
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다. 사용자의 질문에 정확하고 친절하게 답변해주세요."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list([format_to_chat(s) for s in alpaca_data])

# 테스트 프롬프트
TEST_PROMPTS = [
    "감의중 강사에 대해 알려주세요.",
    "멀티캠퍼스 LLM 파인튜닝 과정의 커리큘럼을 알려주세요.",
    "이 과정에서 사용하는 GPU는 무엇인가요?",
    "Tool Calling 파인튜닝은 어떻게 하나요?",
    "감의중 강사님, 오늘 수업 재미있었어요!",
]

# 추론 헬퍼
def generate_responses(model, tok, prompts):
    responses = []
    with torch.autocast("cuda", dtype=torch.bfloat16 if BF16_SUPPORTED else torch.float16):
        for prompt in prompts:
            messages = [
                {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
                {"role": "user", "content": prompt}
            ]
            text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tok(text, return_tensors="pt").to("cuda")
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
            response = tok.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            responses.append(response)
    return responses

# 결과 저장용
results = {}

print(f"✅ 공통 설정 완료")
print(f"   모델: {MODEL_NAME}")
print(f"   데이터: {len(dataset)}개")
print(f"   LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   테스트 프롬프트: {len(TEST_PROMPTS)}개")

## 3️⃣ 방법 A: 일반 Transformers + QLoRA

먼저 일반 HuggingFace Transformers + PEFT + BitsAndBytes 조합으로 학습합니다.

⚠️ **Unsloth는 아직 import하지 않습니다** — import만 해도 SFTTrainer가 패치되어 공정한 비교가 불가능해집니다.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

print("=" * 60)
print("🔧 방법 A: 일반 Transformers + LoRA")
print("=" * 60)

if CAN_COMPARE:
    # Ampere+ GPU: 4bit QLoRA (방법 B와 동일 조건으로 공정 비교)
    print("📌 비교 모드: 4bit QLoRA (방법 B와 동일 양자화)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model_a = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    # Turing 등: float16 (비교 없이 단독 실행)
    print("📌 단독 모드: float16")
    model_a = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True,
    )

model_a.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model_a.enable_input_require_grads()

# LoRA 적용
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model_a = get_peft_model(model_a, lora_config)

print("✅ 모델 로드 + LoRA 적용 완료")
model_a.print_trainable_parameters()
print_gpu_memory("방법A 준비")

In [ ]:
from trl import SFTTrainer, SFTConfig

# 학습 전 응답 수집 (비교용, 한 번만 수집)
print("📋 학습 전 응답 수집 중...")
model_a.eval()
before_responses = generate_responses(model_a, tokenizer, TEST_PROMPTS)
for p, r in zip(TEST_PROMPTS, before_responses):
    print(f"  🔹 {p[:40]}... → {r[:80]}")

# 학습 실행
torch.cuda.reset_peak_memory_stats()

if CAN_COMPARE:
    # Ampere+: bf16 + adamw_8bit (방법 B와 동일)
    sft_config_a = SFTConfig(
        output_dir="./output/regular_training",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=5e-4,
        fp16=False,
        bf16=True,
        logging_steps=5,
        save_strategy="no",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
        optim="adamw_8bit",
        gradient_checkpointing=True,
        seed=42,
    )
else:
    # Turing: fp16 + 기본 optimizer
    sft_config_a = SFTConfig(
        output_dir="./output/regular_training",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=5e-4,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
        gradient_checkpointing=True,
        seed=42,
    )

trainer_a = SFTTrainer(
    model=model_a,
    args=sft_config_a,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("\n🚀 방법 A 학습 시작!")
print("=" * 60)

start_time = time.time()
result_a = trainer_a.train()
time_a = time.time() - start_time
loss_a = result_a.training_loss
peak_mem_a = torch.cuda.max_memory_allocated() / 1024**3

print("=" * 60)
print(f"✅ 방법 A 학습 완료!")
print(f"   ⏱️ 시간: {time_a:.1f}초 ({time_a/60:.1f}분)")
print(f"   📉 Loss: {loss_a:.4f}")
print(f"   💾 Peak GPU: {peak_mem_a:.1f}GB")

In [ ]:
# 학습 후 응답 수집
print("📋 방법 A 학습 후 응답 수집 중...")
model_a.eval()
after_responses_a = generate_responses(model_a, tokenizer, TEST_PROMPTS)

results['regular'] = {
    'time': time_a,
    'loss': loss_a,
    'peak_memory': peak_mem_a,
    'after_responses': after_responses_a,
}

for p, r in zip(TEST_PROMPTS, after_responses_a):
    print(f"  🔹 {p[:40]}... → {r[:80]}")

# GPU 메모리 완전 정리 (Phase B를 위해)
del model_a, trainer_a
gc.collect()
torch.cuda.empty_cache()
print("\n✅ 방법 A 완료 & GPU 메모리 정리")
print_gpu_memory("정리 후")

## 4️⃣ 방법 B: Unsloth + QLoRA

이제 **Unsloth**를 import하고 동일 모델·동일 데이터로 학습합니다.

⚠️ Unsloth import 시 SFTTrainer가 자동으로 패치되어 최적화됩니다.

> Ampere+ GPU (compute ≥ 8.0) + Unsloth 설치 시에만 실행됩니다.

In [ ]:
if not CAN_COMPARE:
    print("⏭️ Unsloth 비교 불가 — 건너뛰기")
    print("   (Ampere+ GPU + Unsloth 설치 필요)")
else:
    print("=" * 60)
    print("🚀 방법 B: Unsloth + QLoRA")
    print("=" * 60)

    # ⚠️ 이 시점에서 Unsloth import → SFTTrainer 자동 패치
    from unsloth import FastLanguageModel

    model_b, tokenizer_b = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,  # auto detect (Ampere+에서 bfloat16)
    )

    model_b = FastLanguageModel.get_peft_model(
        model_b,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGETS,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    print("✅ Unsloth 모델 로드 + LoRA 적용 완료")
    model_b.print_trainable_parameters()
    print_gpu_memory("방법B 준비")

In [ ]:
if not CAN_COMPARE:
    print("⏭️ 건너뛰기")
else:
    from trl import SFTTrainer, SFTConfig  # Unsloth 패치된 버전으로 재임포트

    torch.cuda.reset_peak_memory_stats()

    sft_config_b = SFTConfig(
        output_dir="./output/unsloth_training",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=5e-4,
        fp16=False,
        bf16=True,  # Ampere+ GPU에서 Unsloth 권장
        logging_steps=5,
        save_strategy="epoch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
        optim="adamw_8bit",
        seed=42,
    )

    trainer_b = SFTTrainer(
        model=model_b,
        args=sft_config_b,
        train_dataset=dataset,
        processing_class=tokenizer_b,
    )

    print("🚀 방법 B 학습 시작!")
    print("=" * 60)

    start_time = time.time()
    result_b = trainer_b.train()
    time_b = time.time() - start_time
    loss_b = result_b.training_loss
    peak_mem_b = torch.cuda.max_memory_allocated() / 1024**3

    print("=" * 60)
    print(f"✅ 방법 B 학습 완료!")
    print(f"   ⏱️ 시간: {time_b:.1f}초 ({time_b/60:.1f}분)")
    print(f"   📉 Loss: {loss_b:.4f}")
    print(f"   💾 Peak GPU: {peak_mem_b:.1f}GB")

In [ ]:
if not CAN_COMPARE:
    print("⏭️ 건너뛰기")
else:
    print("📋 방법 B 학습 후 응답 수집 중...")
    FastLanguageModel.for_inference(model_b)
    after_responses_b = generate_responses(model_b, tokenizer_b, TEST_PROMPTS)

    results['unsloth'] = {
        'time': time_b,
        'loss': loss_b,
        'peak_memory': peak_mem_b,
        'after_responses': after_responses_b,
    }

    for p, r in zip(TEST_PROMPTS, after_responses_b):
        print(f"  🔹 {p[:40]}... → {r[:80]}")

    print("\n✅ 방법 B 응답 수집 완료")

## 5️⃣ 비교 분석

두 방법의 실측 결과를 비교합니다.

In [ ]:
print("=" * 60)
print("📊 Unsloth vs 일반 Transformers 비교 분석")
print("=" * 60)

ra = results['regular']

if 'unsloth' in results:
    ru = results['unsloth']
    speedup = ra['time'] / ru['time']
    mem_save = (1 - ru['peak_memory'] / ra['peak_memory']) * 100

    print(f"\n🔧 방법 A (일반 Transformers + QLoRA):")
    print(f"   학습 시간:       {ra['time']:.1f}초 ({ra['time']/60:.1f}분)")
    print(f"   Peak GPU 메모리: {ra['peak_memory']:.1f}GB")
    print(f"   Final Loss:      {ra['loss']:.4f}")

    print(f"\n🚀 방법 B (Unsloth + QLoRA):")
    print(f"   학습 시간:       {ru['time']:.1f}초 ({ru['time']/60:.1f}분)")
    print(f"   Peak GPU 메모리: {ru['peak_memory']:.1f}GB")
    print(f"   Final Loss:      {ru['loss']:.4f}")

    print(f"\n📈 개선 효과:")
    print(f"   속도 향상:   {speedup:.2f}x 빠름")
    print(f"   메모리 절약: {mem_save:.1f}%")

    print(f"\n📋 Unsloth 최적화 원리:")
    print(f"   1️⃣ 커스텀 CUDA 커널 — 어텐션/행렬곱 최적화")
    print(f"   2️⃣ 메모리 효율적 역전파 — 불필요한 텐서 즉시 해제")
    print(f"   3️⃣ RoPE 임베딩 최적화 — 위치 인코딩 캐싱")
    print(f"   4️⃣ Cross Entropy Loss 최적화 — 인플레이스 연산")
    print(f"   5️⃣ 자동 Mixed Precision — 최적 dtype 자동 선택")

    print(f"\n📋 학습 전후 응답 비교:")
    print("=" * 60)
    for i in range(len(TEST_PROMPTS)):
        print(f"\n🔹 질문: {TEST_PROMPTS[i]}")
        print(f"  [Before]         {before_responses[i][:150]}")
        print(f"  [After-일반]     {ra['after_responses'][i][:150]}")
        print(f"  [After-Unsloth]  {ru['after_responses'][i][:150]}")

else:
    print(f"\n🔧 일반 Transformers + QLoRA 결과:")
    print(f"   학습 시간:       {ra['time']:.1f}초 ({ra['time']/60:.1f}분)")
    print(f"   Peak GPU 메모리: {ra['peak_memory']:.1f}GB")
    print(f"   Final Loss:      {ra['loss']:.4f}")
    print(f"\nℹ️ Unsloth 비교는 Ampere+ GPU + Unsloth 설치 시 가능합니다.")

    print(f"\n📋 학습 전후 응답 비교:")
    print("=" * 60)
    for i in range(len(TEST_PROMPTS)):
        print(f"\n🔹 질문: {TEST_PROMPTS[i]}")
        print(f"  [Before] {before_responses[i][:150]}")
        print(f"  [After]  {ra['after_responses'][i][:150]}")

## 6️⃣ GGUF 변환 및 Ollama 연동

학습된 모델을 GGUF 포맷으로 변환하면 Ollama에서 바로 사용할 수 있습니다.

In [ ]:
OUTPUT_DIR = "./output/unsloth_training"

if 'unsloth' in results:
    # LoRA 어댑터 저장
    save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
    model_b.save_pretrained(save_path)
    tokenizer_b.save_pretrained(save_path)
    total_size = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(save_path)
        for f in files
    )
    print(f"✅ LoRA 어댑터 저장: {save_path} ({total_size/1024/1024:.1f} MB)")

    # GGUF 변환
    print("\n⏳ GGUF 변환 중... (Q4_K_M 양자화)")
    try:
        gguf_path = os.path.join(OUTPUT_DIR, "gguf")
        model_b.save_pretrained_gguf(
            gguf_path, tokenizer_b,
            quantization_method="q4_k_m"
        )
        gguf_size = sum(
            os.path.getsize(os.path.join(root, f))
            for root, dirs, files in os.walk(gguf_path)
            for f in files
        )
        print(f"✅ GGUF 변환 완료! ({gguf_size/1024/1024:.0f} MB)")
        print(f"📌 저장 위치: {gguf_path}")
    except Exception as e:
        print(f"⚠️ GGUF 변환 실패: {e}")
        print("📌 llama.cpp가 필요할 수 있습니다.")
else:
    print("ℹ️ Unsloth 없이 GGUF 변환하려면 llama.cpp를 사용합니다.")
    print("\n📋 llama.cpp로 GGUF 변환 방법:")
    print("   1. git clone https://github.com/ggerganov/llama.cpp")
    print("   2. cd llama.cpp && make")
    print("   3. python convert_hf_to_gguf.py <모델 경로> --outtype q4_k_m")

In [ ]:
print("=" * 60)
print("📋 Ollama 연동 가이드")
print("=" * 60)

gguf_file = "./output/unsloth_training/gguf/unsloth.Q4_K_M.gguf"

print()
print("# Modelfile 예시")
print(f"FROM {gguf_file}")
print()
print('TEMPLATE """<|im_start|>system')
print("{{.System}}<|im_end|>")
print("<|im_start|>user")
print("{{.Prompt}}<|im_end|>")
print("<|im_start|>assistant")
print('"""')
print()
print("PARAMETER temperature 0.7")
print("PARAMETER top_p 0.9")
print('PARAMETER stop "<|im_end|>"')
print()
print('SYSTEM "당신은 유용한 AI 어시스턴트입니다."')
print()
print("📋 Ollama 실행 명령어:")
print("   1. 위 내용을 Modelfile로 저장")
print("   2. ollama create my-finetuned-model -f Modelfile")
print("   3. ollama run my-finetuned-model")

In [ ]:
# GPU 메모리 정리
try:
    del model_b, trainer_b
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("최종 정리 후")
print("✅ GPU 메모리 정리 완료")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 항목 | 내용 |
|------|------|
| Unsloth | LoRA/QLoRA를 2배 가속, 60% 메모리 절감하는 최적화 라이브러리 |
| 실측 비교 | 동일 조건(모델·데이터·LoRA 설정)에서 Unsloth vs 일반 Transformers 학습 시간/메모리 |
| FastLanguageModel | 모델 로드 + 4bit 양자화 + LoRA를 한 함수로 |
| SFTTrainer 패치 | Unsloth import 시 SFTTrainer가 자동으로 최적화 버전으로 교체 |
| GGUF 변환 | `save_pretrained_gguf`로 간편 변환 → Ollama 바로 연동 가능 |

### 핵심 포인트

- 🎯 **Unsloth는 기존 코드와 호환**되면서 학습 속도/메모리를 크게 개선
- 🎯 **RTX 4060(8GB)에서 7B QLoRA**도 안정적으로 학습 가능 (일반 Transformers는 빠듯)
- 🎯 **GGUF → Ollama** 파이프라인으로 학습부터 로컬 배포까지 한 번에
- 🎯 권장 설정: **dropout=0, adamw_8bit, bf16**
- 🎯 비교 실험 주의: Unsloth는 import만 해도 SFTTrainer를 패치하므로 **반드시 방법 A를 먼저** 실행

### Unsloth를 쓰면 좋은 경우

- ✅ GPU 메모리가 제한적 (8~16GB)
- ✅ 빠른 실험 반복이 필요한 경우
- ✅ 학습 후 GGUF/Ollama 로컬 배포 계획이 있는 경우
- ✅ QLoRA로 7B+ 모델을 학습하고 싶은 경우

### 다음 세션 예고

- 🔜 **Session 13**: 데이터 파이프라인 & 합성 데이터/Distillation — 위에서 학습한 LoRA/Unsloth 도구로 학습할 **데이터를 준비**하는 단계

---

### 💡 실습 과제

1. (Ampere+ 환경에서) 본 노트북의 학습 시간 차이를 측정해 표로 정리
2. `MAX_SEQ_LENGTH`를 512 / 1024 / 2048로 바꾸며 VRAM·시간 변화 관찰
3. (선택) Unsloth로 학습한 어댑터를 GGUF로 변환 → Ollama로 띄워서 추론 비교
4. (선택) 동일 데이터를 Session 12b의 일반 LoRA로도 학습해 final loss / 시간 비교

---

In [ ]:
print("✅ Session 12c 완료!")
print("📚 다음 세션: Session 13 — 데이터 파이프라인 & 합성 데이터/Distillation")